In [ ]:
# ======================= INFERENCE & POST-PROCESSING =======================

def predict_large_image(model, image_path, output_dir, patch_size=512, overlap=52):
    os.makedirs(output_dir, exist_ok=True)
    patch_gen = PatchGenerator(patch_size, overlap)
    
    with rasterio.open(image_path) as src:
        img = src.read().transpose(1, 2, 0)
        meta = src.meta.copy()
        meta.update(dtype=rasterio.uint8, count=1, nodata=0)
        
        # Convert to float and normalize
        img = img.astype(np.float32) / 65535.0
        
        # Generate patches
        patches, positions = patch_gen.split_image(img)
    
    # Predict in batches
    predictions = []
    batch_size = 4
    
    for i in range(0, len(patches), batch_size):
        batch = patches[i:i+batch_size]
        preds = model.predict(batch, verbose=0)
        predictions.extend(preds)
    
    # Stitch predictions
    class_map = np.argmax(np.array(predictions), axis=-1)
    stitched = patch_gen.stitch_patches(
        class_map, 
        positions, 
        (img.shape[0], img.shape[1], 1)
    )
    stitched = np.squeeze(stitched).astype(np.uint8)
    
    # Save class map
    class_map_path = os.path.join(output_dir, 'class_map.tif')
    with rasterio.open(class_map_path, 'w', **meta) as dst:
        dst.write(stitched, 1)
    
    # Generate masks and shapefiles
    generate_shapefiles(stitched, meta, output_dir)
    
    return class_map_path

def generate_shapefiles(class_map, meta, output_dir):
    # Create masks
    cloud_mask = (class_map == 1).astype(np.uint8)
    shadow_mask = (class_map == 2).astype(np.uint8)
    
    # Save mask files
    with rasterio.open(os.path.join(output_dir, 'cloud_mask.tif'), 'w', **meta) as dst:
        dst.write(cloud_mask, 1)
    
    with rasterio.open(os.path.join(output_dir, 'shadow_mask.tif'), 'w', **meta) as dst:
        dst.write(shadow_mask, 1)
    
    # Generate polygons in parallel
    with Pool() as pool:
        cloud_polygons = pool.starmap(extract_polygons, [(cloud_mask, meta['transform'])])
        shadow_polygons = pool.starmap(extract_polygons, [(shadow_mask, meta['transform'])])
    
    # Save shapefiles
    save_shapefile(cloud_polygons[0], os.path.join(output_dir, 'cloud_polygons.shp'), 'cloud')
    save_shapefile(shadow_polygons[0], os.path.join(output_dir, 'shadow_polygons.shp'), 'shadow')

def extract_polygons(mask, transform):
    return [
        (shape(geom), value)
        for geom, value in shapes(mask, transform=transform)
        if value == 1
    ]

def save_shapefile(polygons, path, feature_type):
    schema = {
        'geometry': 'Polygon',
        'properties': {'class': 'int'}
    }
    
    with fiona.open(path, 'w', 
                   driver='ESRI Shapefile',
                   schema=schema,
                   crs=from_epsg(4326)) as dst:
        for geom, value in polygons:
            dst.write({
                'geometry': geom.__geo_interface__,
                'properties': {'class': value}
            })
